# Clasificación de textos sobre Túpac Amaru II: tema y encuadre

Clasifica fragmentos del corpus (`data/tupac_amaru_corpus.csv`) **sin entrenar** (*zero-shot*, modelo NLI multilingüe)
y deja columnas para que **tú valides a mano**.

## Por qué dos ejes y no seis categorías

Las seis categorías (causas, rebelión, revolución, juicio, legado, influencia) responden a **dos preguntas distintas**:

| Eje | Pregunta | Valores |
|---|---|---|
| **Tema** | ¿De qué trata el fragmento? | causas · hechos · juicio · legado · influencia · otro |
| **Encuadre** | ¿Cómo caracteriza el autor al movimiento? | rebelión · revolución · neutral |

*Rebelión* vs. *revolución* es un **debate historiográfico**: unos autores hablan de rebelión porque no triunfó o
no buscó cambios sustanciales; otros de revolución porque proponía transformar el orden colonial. Un mismo fragmento
sobre las *causas* puede estar encuadrado como *rebelión* o como *revolución*, así que se miden por separado.

**Advertencias metodológicas**
- Las fuentes de época (volúmenes del Congreso) están escritas por autoridades coloniales: dicen *rebelión*
  por posición política, no por una tesis historiográfica. El encuadre tiene más sentido en los **artículos académicos**.
- El encuadre vive sobre todo **a nivel de autor/documento**, no de fragmento. Por eso hay un resumen por documento
  y un conteo de términos (transparente y verificable) junto al modelo.
- Las definiciones son una **propuesta**: edítalas.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import torch
from transformers import pipeline

DATA = Path("data")
CORPUS = DATA / "tupac_amaru_corpus.csv"
MUESTRA_POR_DOCUMENTO = 40      # sube este número para clasificar más; None = todo (muy lento en CPU)
MAX_CHARS = 1000                # tamaño máximo de cada fragmento
SEED = 42
print("CUDA:", torch.cuda.is_available())

## 1. Ejes y definiciones (editables)

In [ ]:
TEMAS = {
    "causas": "las causas de la rebelión: agravios, abusos de corregidores, reparto, mita, impuestos, "
              "reformas borbónicas y antecedentes que explican por qué estalló",
    "hechos": "los hechos del levantamiento: movilización de gente, batallas, cercos, marchas, "
              "operaciones militares y enfrentamientos entre rebeldes y realistas",
    "juicio": "la captura, el proceso judicial, los interrogatorios, la sentencia y la ejecución "
              "de Túpac Amaru y sus allegados",
    "legado": "la memoria, el recuerdo y la herencia histórica de Túpac Amaru: conmemoraciones, "
              "símbolos, homenajes y cómo se le recuerda después",
    "influencia": "la influencia y repercusiones del movimiento en rebeliones posteriores, "
                  "en la independencia y en la historia política del Perú y América",
    "otro": "asuntos administrativos, listas, cuentas, correspondencia rutinaria u otros temas "
            "que no tienen relación con la rebelión de Túpac Amaru",
}
ENCUADRES = {
    "rebelión": "una rebelión o levantamiento contra abusos, que reclama reformas o busca restaurar un orden "
                "justo, sin proponer cambiar el sistema colonial",
    "revolución": "una revolución que busca transformar o reemplazar el orden colonial, con un proyecto "
                  "político y social propio",
    "neutral": "hechos o documentos que no caracterizan el movimiento ni como rebelión ni como revolución",
}
CLASES = {"tema": TEMAS, "encuadre": ENCUADRES}
pd.concat({k: pd.Series(v) for k, v in CLASES.items()}).to_frame("definición")

## 2. Cargar el corpus y cortarlo en fragmentos

In [ ]:
def trocear(texto: str, max_chars: int = MAX_CHARS, min_chars: int = 250) -> list[str]:
    """Corta en fragmentos de <= max_chars respetando, si se puede, el final de oración."""
    out, resto = [], texto.strip()
    while len(resto) > max_chars:
        corte = max(resto.rfind(". ", 0, max_chars), resto.rfind("; ", 0, max_chars))
        corte = corte + 1 if corte > max_chars * 0.5 else max_chars
        out.append(resto[:corte].strip())
        resto = resto[corte:].strip()
    if resto:
        out.append(resto)
    return [t for t in out if len(t) >= min_chars]

corpus = pd.read_csv(CORPUS, encoding="utf-8-sig")
# Solo fuentes primarias y artículos: se excluye el frontispicio/colofón editorial (columna `rol`).
# `texto_fuente` es el cuerpo de la página sin notas al pie de los editores.
corpus = corpus[corpus.rol != "paratexto_editorial"]
filas = [
    {"documento": r.documento, "pagina": r.pagina, "url": r.url, "texto": t}
    for r in corpus.itertuples() for t in trocear(r.texto_fuente)
]
frag = pd.DataFrame(filas)
print(f"{len(corpus)} páginas -> {len(frag)} fragmentos")
frag.groupby("documento").size().to_frame("fragmentos")

## 3. Muestra estratificada por documento

Se toma el mismo número de fragmentos de cada documento para que los volúmenes más largos no dominen.

In [ ]:
if MUESTRA_POR_DOCUMENTO:
    muestra = (frag.sample(frac=1, random_state=SEED)          # mezcla
                   .groupby("documento").head(MUESTRA_POR_DOCUMENTO)
                   .sort_values(["documento", "pagina"]).reset_index(drop=True))
else:
    muestra = frag.copy()
print(len(muestra), "fragmentos a clasificar")

## 4. Modelo zero-shot

Se usa `mDeBERTa-v3-base-mnli-xnli` (multilingüe, incluye español). Si no está en la carpeta local, se descarga del Hub.

In [ ]:
LOCAL = Path.home() / ".cache" / "tupac_models" / "mdeberta-xnli"
MODELO = str(LOCAL) if (LOCAL / "model.safetensors").exists() else "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
print("Modelo:", MODELO)

clf = pipeline("zero-shot-classification", model=MODELO,
               device=0 if torch.cuda.is_available() else -1)

In [ ]:
import time

def clasificar(textos: list[str], eje: str) -> pd.DataFrame:
    # Clasifica con las definiciones del eje ('tema' o 'encuadre'); devuelve etiqueta, confianza y 2ª opción.
    etiquetas = list(CLASES[eje]); descr = list(CLASES[eje].values())
    plantilla = ("Este fragmento trata sobre {}." if eje == "tema"
                 else "Este fragmento presenta el movimiento como {}.")
    res = clf(textos, candidate_labels=descr, hypothesis_template=plantilla, batch_size=8, truncation=True)
    filas = []
    for r in res:
        orden = [etiquetas[descr.index(d)] for d in r["labels"]]
        filas.append({f"{eje}_modelo": orden[0], f"{eje}_conf": round(r["scores"][0], 3),
                      f"{eje}_2a": orden[1]})
    return pd.DataFrame(filas)

t0 = time.time()
prueba = clasificar(muestra.texto.head(8).tolist(), "tema")
seg = (time.time() - t0) / 8
print(f"{seg:.1f} s por fragmento y eje -> ~{2 * seg * len(muestra) / 60:.0f} min para {len(muestra)} fragmentos (2 ejes)")
prueba

## 5. Clasificar la muestra y exportar para validación manual

In [ ]:
textos = muestra.texto.tolist()
salida = pd.concat([muestra, clasificar(textos, "tema"), clasificar(textos, "encuadre")], axis=1)

# Señal léxica, transparente: ¿qué palabras usa el texto?
RE_REBELION = re.compile(r"rebeli[óo]n|rebelde|levantamiento|sublevaci[óo]n|alzamiento|sedici[óo]n|insurrecci[óo]n", re.I)
RE_REVOLUCION = re.compile(r"revoluci[óo]n|revolucionari|tupamari|utop[íi]a|proyecto (?:pol[íi]tico|social)|nuevo orden", re.I)
salida["n_rebelion"] = salida.texto.map(lambda t: len(RE_REBELION.findall(t)))
salida["n_revolucion"] = salida.texto.map(lambda t: len(RE_REVOLUCION.findall(t)))

salida["tema_manual"] = ""        # <- rellena en Excel: causas / hechos / juicio / legado / influencia / otro
salida["encuadre_manual"] = ""    # <- rellena en Excel: rebelión / revolución / neutral
destino = DATA / "clasificacion_muestra.csv"
salida.to_csv(destino, index=False, encoding="utf-8-sig")
print("Guardado en", destino)
pd.concat([salida.tema_modelo.value_counts(), salida.encuadre_modelo.value_counts()], axis=1)

In [ ]:
# Resumen por documento: el encuadre vive a nivel de autor/obra
resumen = salida.assign(doc=salida.documento.str.slice(0, 60)).groupby("doc").agg(
    fragmentos=("texto", "size"),
    pct_encuadre_revolucion=("encuadre_modelo", lambda s: round((s == "revolución").mean() * 100)),
    pct_encuadre_rebelion=("encuadre_modelo", lambda s: round((s == "rebelión").mean() * 100)),
    menciones_rebelion=("n_rebelion", "sum"),
    menciones_revolucion=("n_revolucion", "sum"),
)
resumen

In [ ]:
# Ejemplos para revisar a ojo: los más seguros de cada tema
for cat in TEMAS:
    sub = salida[salida.tema_modelo == cat].nlargest(2, "tema_conf")
    for r in sub.itertuples():
        print(f"[{cat} | {r.tema_conf}] {r.documento[:40]} p.{r.pagina} | encuadre: {r.encuadre_modelo}")
        print(f"  {r.texto[:350]}...")
        print()

## 6. Validación manual

1. Abre `data/clasificacion_muestra.csv` en Excel.
2. Para unas 60–100 filas rellena **`tema_manual`** (`causas`, `hechos`, `juicio`, `legado`, `influencia`, `otro`)
   y **`encuadre_manual`** (`rebelión`, `revolución`, `neutral`). Déjalas vacías si dudas.
3. Guarda el CSV y ejecuta la celda siguiente.

*Anota también los casos dudosos: esos son los que muestran dónde la categoría está mal definida.*

In [ ]:
from sklearn.metrics import classification_report

val_all = pd.read_csv(DATA / "clasificacion_muestra.csv", encoding="utf-8-sig")
for eje in ["tema", "encuadre"]:
    val = val_all[val_all[f"{eje}_manual"].isin(CLASES[eje])]
    print()
    print(f"===== {eje.upper()}: {len(val)} fragmentos validados a mano =====")
    if len(val):
        print("Acierto global:", round((val[f"{eje}_manual"] == val[f"{eje}_modelo"]).mean(), 2))
        print(classification_report(val[f"{eje}_manual"], val[f"{eje}_modelo"], zero_division=0))
        print(pd.crosstab(val[f"{eje}_manual"], val[f"{eje}_modelo"], rownames=["tú"], colnames=["modelo"]))

## Siguiente paso: fine-tuning

Si el acierto zero-shot no es suficiente, usa tus `etiqueta_manual` (idealmente 30–50 por categoría)
para **entrenar** un modelo en español (por ejemplo `dccuchile/bert-base-spanish-wwm-cased`) con el mismo
esquema de la clase 9 (`lecture-modernbert.ipynb`), cambiando el dataset y el `model_id`.